# PerfectPunch v25 — Five Large Skeleton-Action Models · 5-Fold CV

**Goal:** Compare exactly five task-adapted, from-scratch large models: `dinat_large`, `stgcn_large`, `semgcn_large`, `motionbert_large`, and `dastformer_large`. All accept the existing seven-channel `(B, 7, 50, 4, 3)` skeleton volume and return three logits ordered `jab`, `lead-hook`, `rear-uppercut`. No legacy small CNN, TCN, ST-GCN, DiNAT, or generic Transformer models are included.

**Important naming distinction:** These are **large relative to the previous PerfectPunch baselines**, not the original published ImageNet-sized DiNAT-Large or official pretrained MotionBERT/DASTFormer checkpoints. They implement the cited architectural ideas adapted to **four arm joints**, 50 frames, and three-class action classification; they are not paper-exact replications.

**Preprocessing contract unchanged:** MediaPipe Tasks Pose Landmarker **Lite (float16)**, 3D world landmarks, hip-centered and shoulder-width-normalized. Convert to 60 FPS, select 50 frames ending at the smoothed 4-joint peak (`VELOCITY_LM=[13,14,15,16]`, `VEL_WINDOW=5`), then construct the same seven channels (ch5 uses full XYZ speed). If the recording has fewer than 50 resampled frames, first-frame front-pad it. Data preparation remains v25-compatible; only the model zoo and output run directory change.

**Evaluation:** All five candidates use the same five stratified **video-level** held-out folds. Splits are **not participant-disjoint**. Training-only spatial/temporal augmentation and 30→60 FPS webcam proxy remain available. Large models may overfit a 120-video dataset or take substantially longer to train; stronger results are not guaranteed. `RUN_EXPERIMENTS=False` and `RUN_FINAL_REFIT=False` initially so that opening/running this notebook does not silently launch 25 large fold jobs. Enable full CV explicitly after confirming data access and GPU capacity.

**Deployment:** Final input `[1,7,50,4,3]`, output `[1,3]`; exported ONNX names default to `skeleton_volume` / `class_logits`. Attention models are **not yet verified on ONNX Runtime Web**; test browser provider support and numerical parity before shipping any exported weights. Model weights from earlier architectures are not interchangeable with these new definitions.


## 0. Install & mount Google Drive

In [40]:
!pip install -q "mediapipe>=0.10.30"   # 0.10.13 is no longer available on Colab
!pip install -q onnx onnxscript

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Global configuration — check these paths before running segmentation

In [41]:
import os, glob, json, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold, train_test_split
import matplotlib.pyplot as plt

SEED = 42
CLIP_FRAMES = 50
TARGET_FPS = 60  # JS pipeline target FPS (30 FPS camera samples may be interpolated)
N_JOINTS = 33
CLASS_NAMES = ['jab', 'lead-hook', 'rear-uppercut']
N_CLASSES = len(CLASS_NAMES)
N_CHANNELS = 7
ARM_JOINTS = [13, 14, 15, 16]
N_ARM_JOINTS = len(ARM_JOINTS)
VELOCITY_LM = [13, 14, 15, 16]  # both elbows + both wrists, matching agreed JS contract
VEL_WINDOW = 5

# The team's shared cloud folder must first be accessible at this mounted path.
# Edit this line to match your Google Drive shortcut location.
POSE_VARIANT = 'lite'  # preprocessing AND live webcam; 'lite' | 'full' | 'heavy'
assert POSE_VARIANT in ('lite', 'full', 'heavy')
INPUT_FOLDER = '/content/drive/MyDrive/newPerfectPunchData/punch_dataset'
DRIVE_OUTPUT_BASE = '/content/drive/MyDrive/newPerfectPunchData/processed_data_lite_peak_v25'
DRIVE_NPY = os.path.join(DRIVE_OUTPUT_BASE, 'npy_data')
RUN_DIR = '/content/drive/MyDrive/newPerfectPunchData/stratified5_lite_peak_v25_large5'
CLASS_FOLDERS = {
    'jab': ['jab'],
    'lead-hook': ['hook', 'lead-hook'],
    'rear-uppercut': ['uppercut', 'rear-uppercut'],
}
VIDEO_EXTENSIONS = ('.mp4', '.mov', '.avi', '.m4v')

def class_folder(root, label):
    found = [Path(root)/name for name in CLASS_FOLDERS[label] if (Path(root)/name).is_dir()]
    if len(found) > 1:
        raise ValueError(f'Ambiguous source folders for {label}: {found}. Keep just one of the aliases.')
    return str(found[0]) if found else None

def source_videos(folder):
    return sorted([p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS],
                  key=lambda p: p.name.lower())

# Processing: False for quick re-runs after the .npy files exist.
AUTO_SEGMENT = True
FORCE_REPROCESS = False  # v25 uses a NEW folder; True to redo its existing .npy files

# Training-only augmentation. `v12` is identical to the uploaded source.
AUGMENTATION_MODE = 'v12'  # 'v12', 'improved', or 'none'
HOOK_SPECIFIC_AUG = False  # only in improved mode
MIRROR_PROB = 0.5          # only in improved mode
AUGMENT_TRAIN = True
# Training-only browser-style FPS degradation on segmented 50-frame clips.
# Kept separate from spatial/time-warp augmentation; NEVER applied to validation/test.
WEBCAM_FPS_AUG_PROB = 0.50
WEBCAM_CAPTURE_FPS = 30
assert 0 <= WEBCAM_FPS_AUG_PROB <= 1
PREPROCESS_CONTRACT = 'mediapipe_lite_peak4j_smooth5_60fps_50frames_ch5_3dspeed_v25'
SHORT_CLIP_POLICY = 'front_pad'  # JS currently returns null for <50 frames; see checklist
SAMPLER_MODE = 'hook_x3'   # 'hook_x3', 'balanced', 'shuffle'

N_SPLITS = 5  # True stratified 5-fold CV: ~20% test per fold, balanced by punch label
VAL_FRACTION = 0.15  # fraction of the remaining 80%, not of the full dataset
MAX_FOLDS = None  # None = all 5 folds; e.g. 2 for a quick debug run
MODEL_NAMES = [
    'dinat_large',
    'stgcn_large',
    'semgcn_large',
    'motionbert_large',
    'dastformer_large',
]
MODELS_TO_RUN = MODEL_NAMES.copy()  # exactly these five model variants

EPOCHS, PATIENCE = 120, 20
BATCH_SIZE, LR, WEIGHT_DECAY = 8, 2e-4, 1e-2  # smaller batches / stronger decay for large models
NUM_WORKERS = 0
RUN_EXPERIMENTS = False  # opt in to 5 models x 5 held-out folds (25 large trainings)
FINAL_MODEL_NAME = 'stgcn_large'  # placeholder; choose AFTER reviewing complete CV metrics
RUN_FINAL_REFIT = True  # enable ONLY after complete CV + explicit model selection

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
assert os.path.isdir(INPUT_FOLDER), f'Raw videos not found: {INPUT_FOLDER}. Check the shared Drive shortcut/path.'
os.makedirs(DRIVE_NPY, exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)
for label in CLASS_NAMES:
    folder = class_folder(INPUT_FOLDER, label)
    print(f'{label:<15} <- {folder} ({len(source_videos(folder)) if folder else 0} videos)')
    if folder is None:
        raise FileNotFoundError(f'Missing source folder for {label}: expected one of {CLASS_FOLDERS[label]}')
print('Config ready | pose:', POSE_VARIANT, '| features:', N_CHANNELS, '| FPS:', TARGET_FPS, '| RAW:', INPUT_FOLDER)
print('Processed v25 peak clips:', DRIVE_NPY, '| v25 CV results:', RUN_DIR)


jab             <- /content/drive/MyDrive/newPerfectPunchData/punch_dataset/jab (40 videos)
lead-hook       <- /content/drive/MyDrive/newPerfectPunchData/punch_dataset/hook (40 videos)
rear-uppercut   <- /content/drive/MyDrive/newPerfectPunchData/punch_dataset/uppercut (40 videos)
Config ready | pose: lite | features: 7 | FPS: 60 | RAW: /content/drive/MyDrive/newPerfectPunchData/punch_dataset
Processed v25 peak clips: /content/drive/MyDrive/newPerfectPunchData/processed_data_lite_peak_v25/npy_data | v25 CV results: /content/drive/MyDrive/newPerfectPunchData/stratified5_lite_peak_v25_large5


## 2. MediaPipe Lite skeletons → 60 FPS → **peak-velocity** 50-frame clips

The raw video is passed through MediaPipe Tasks Pose Landmarker **Lite, VIDEO mode**, using world 3D coordinates. The original JS resampling convention (`linspace` endpoint interpolation) is preserved. After resampling, the compound speed of **four joints** is smoothed using a centered **5-frame** moving average. The clip consists of frames `[peak-49 : peak+1]` (end exclusive), including the peak. No search for reversal or retraction is performed. For source videos shorter than 50 resampled valid landmarks, the offline extractor pads at the front; check the mismatch with the JS `extractLiveClip` short-clip behavior.


In [42]:
import os
import cv2
import mediapipe as mp
import numpy as np
import glob

import urllib.request
from mediapipe.tasks.python import BaseOptions, vision

# mediapipe >= 0.10.30 removed mp.solutions, so pose runs through the Tasks
# PoseLandmarker. Keep extraction model consistent with the JS deployment.
POSE_TASK = f'/content/pose_landmarker_{POSE_VARIANT}.task'
if not os.path.exists(POSE_TASK):
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/pose_landmarker/'
        f'pose_landmarker_{POSE_VARIANT}/float16/latest/'
        f'pose_landmarker_{POSE_VARIANT}.task', POSE_TASK)


def make_pose_landmarker():
    return vision.PoseLandmarker.create_from_options(vision.PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=POSE_TASK),
        running_mode=vision.RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ))

L_WRIST_IDX, R_WRIST_IDX = 15, 16
L_ELBOW_IDX, R_ELBOW_IDX = 13, 14


def normalize_skeleton(world_lm_list):
    """
    Hip-centred, shoulder-width-normalised from pose_world_landmarks.
    World frame: x=lateral, y=vertical (up), z=depth toward camera.
    Returns (33, 4) — [x, y, z, 1.0].
    """
    coords = np.array([[l.x, l.y, l.z] for l in world_lm_list], dtype=np.float32)
    hip_c  = (coords[23] + coords[24]) / 2.0
    coords -= hip_c
    dist   = np.linalg.norm(coords[11] - coords[12])
    if dist > 1e-6:
        coords /= dist
    vis = np.ones((33, 1), dtype=np.float32)
    return np.concatenate([coords, vis], axis=1)


def smooth_signal(arr, window=VEL_WINDOW):
    kernel = np.ones(window) / window
    return np.convolve(arr, kernel, mode='same')


def resample_frames(frames, src_fps, dst_fps=TARGET_FPS):
    """Temporally resample skeleton frames from src_fps to dst_fps (copied from the app)."""
    n_src = len(frames)
    if n_src < 2 or abs(src_fps - dst_fps) < 0.5:
        return np.array(frames, dtype=np.float32)
    arr   = np.array(frames, dtype=np.float32)       # (N, 33, 4)
    dur   = n_src / src_fps
    n_dst = max(2, int(round(dur * dst_fps)))
    t_src = np.linspace(0, 1, n_src)
    t_dst = np.linspace(0, 1, n_dst)
    out   = np.zeros((n_dst, arr.shape[1], arr.shape[2]), dtype=np.float32)
    for j in range(arr.shape[1]):
        for c in range(arr.shape[2]):
            out[:, j, c] = np.interp(t_dst, t_src, arr[:, j, c])
    return out


def read_video_frames(video_path):
    """
    Read every detected-pose frame, then resample to TARGET_FPS.
    Returns (frames, TARGET_FPS).
    """
    cap     = cv2.VideoCapture(video_path)
    src_fps = cap.get(cv2.CAP_PROP_FPS)
    if src_fps <= 0 or src_fps > 500:
        src_fps = 30.0

    frames = []
    # Fresh landmarker per video: VIDEO mode needs increasing timestamps
    with make_pose_landmarker() as landmarker:
        i = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            rgb = np.ascontiguousarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            res = landmarker.detect_for_video(
                mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), int(i * 1000 / src_fps))
            i += 1
            if res.pose_world_landmarks and res.pose_landmarks:
                frames.append(normalize_skeleton(res.pose_world_landmarks[0]))
    cap.release()
    if len(frames) == 0:
        return [], TARGET_FPS
    return resample_frames(frames, src_fps, TARGET_FPS), TARGET_FPS


def find_peak_velocity_frame(frames):
    """Return the maximum smoothed four-joint speed, exactly 50-frame aligned.

    Use elbows 13/14 and wrists 15/16, RMS of per-frame 3D displacements,
    and centered zero-padded 5-frame moving average. Like the agreed JS
    algorithm, for >=50 frames only consider peaks at or after index 49.
    Ties choose the earliest index. No motion reversal is searched.
    """
    xyz = np.asarray(frames, dtype=np.float32)[:, :, :3]
    n = len(xyz)
    if n < 1:
        raise ValueError('Cannot find peak from empty sequence')
    movement = np.diff(xyz[:, VELOCITY_LM, :], axis=0, prepend=xyz[:1, VELOCITY_LM, :])
    speed = np.sqrt(np.mean(np.sum(movement**2, axis=-1), axis=1))
    # Explicit centered zero-padding for consistent n even for short sequences.
    pad = VEL_WINDOW // 2
    smooth = np.convolve(np.pad(speed, (pad, pad)),
                         np.ones(VEL_WINDOW, dtype=np.float64) / VEL_WINDOW,
                         mode='valid')
    min_peak = (CLIP_FRAMES - 1) if n >= CLIP_FRAMES else 0
    peak = min_peak + int(np.argmax(smooth[min_peak:]))
    return peak, float(smooth[peak])


def extract_best_clip(video_path):
    """Segment 50 normalized 60-FPS frames ending AT the speed peak."""
    frames, fps = read_video_frames(video_path)
    if len(frames) == 0:
        print('      No valid pose; skipping')
        return None
    peak, peak_speed = find_peak_velocity_frame(frames)
    end = peak + 1
    start = end - CLIP_FRAMES
    short = max(0, -start)
    if short:
        if SHORT_CLIP_POLICY != 'front_pad':
            raise ValueError(f'Unsupported short-clip policy: {SHORT_CLIP_POLICY}')
        padded = np.repeat(np.asarray(frames[:1], dtype=np.float32), short, axis=0)
        clip = np.concatenate([padded, np.asarray(frames[:end], dtype=np.float32)], axis=0)
    else:
        clip = np.asarray(frames[start:end], dtype=np.float32)
    assert clip.shape == (CLIP_FRAMES, N_JOINTS, 4), clip.shape
    print(f'      fps={fps:.1f}  peak@{peak}  speed={peak_speed:.5f}  '
          f'window[{max(0,start)}:{end}]  front_pad={short}  frames_total={len(frames)}')
    return clip


In [43]:
def segment_all(force=FORCE_REPROCESS):
    """One original video -> one .npy; unchanged extractor, explicit collision checks."""
    contract_file = Path(DRIVE_OUTPUT_BASE) / 'preprocessing_contract.json'
    expected = {
        'id': PREPROCESS_CONTRACT,
        'pose_variant': POSE_VARIANT,
        'fps': TARGET_FPS,
        'clip_frames': CLIP_FRAMES,
        'velocity_landmarks': VELOCITY_LM,
        'velocity_window': VEL_WINDOW,
        'selector': 'peak_inclusive',
        'short_clip_policy': SHORT_CLIP_POLICY,
    }
    cached = list(Path(DRIVE_NPY).glob('*/*.npy'))
    if contract_file.exists() and not force:
        current = json.loads(contract_file.read_text())
        if current != expected:
            raise RuntimeError(f'Cached clips have a different preprocessing contract. Set FORCE_REPROCESS=True. {contract_file}')
    elif cached and not force:
        raise RuntimeError(f'Existing clips without a v25 contract marker: {DRIVE_NPY}. Set FORCE_REPROCESS=True.')
    n_saved, n_reused, n_failed = 0, 0, 0
    for label in CLASS_NAMES:
        src = class_folder(INPUT_FOLDER, label)
        dst = Path(DRIVE_NPY) / label
        dst.mkdir(parents=True, exist_ok=True)
        videos = source_videos(src)
        stems = [p.stem.lower() for p in videos]
        if len(stems) != len(set(stems)):
            raise ValueError(f'Duplicate video stems in {src}; .mp4 and .mov can overwrite the same .npy')
        print(f'\n{label}: {len(videos)} source videos in {src}')
        for video in videos:
            target = dst / (video.stem + '.npy')
            if target.is_file() and not force:
                n_reused += 1
                continue
            clip = extract_best_clip(str(video))
            if clip is None:
                n_failed += 1
                print('  No detected skeleton:', video)
                continue
            assert clip.shape == (CLIP_FRAMES, N_JOINTS, 4), (video, clip.shape)
            np.save(target, clip.astype(np.float32))
            n_saved += 1
    if n_failed:
        raise RuntimeError(f'{n_failed} source videos had no valid MediaPipe pose. Fix/review those videos before five-fold CV.')
    contract_file.write_text(json.dumps(expected, indent=2))
    print(f'Segmentation: saved={n_saved} reused={n_reused} failed={n_failed} | {DRIVE_NPY}')

if AUTO_SEGMENT:
    segment_all()


jab: 40 source videos in /content/drive/MyDrive/newPerfectPunchData/punch_dataset/jab

lead-hook: 40 source videos in /content/drive/MyDrive/newPerfectPunchData/punch_dataset/hook

rear-uppercut: 40 source videos in /content/drive/MyDrive/newPerfectPunchData/punch_dataset/uppercut
Segmentation: saved=0 reused=120 failed=0 | /content/drive/MyDrive/newPerfectPunchData/processed_data_lite_peak_v25/npy_data


## 3. Feature engineering — exactly the uploaded seven channels

The `build_tensor()` below is **byte-for-byte the same Python function** as in `Copy_of_newPerfectPunchModel.ipynb`. It outputs `(7,T,4,3)` with position, velocity, geometry, y-velocity, elbow angle, lateral dominance, and vertical dominance. No model-specific feature changes; deployed inference uses this function too.

In [44]:
import numpy as np

L_SHOULDER, R_SHOULDER = 11, 12
L_ELBOW,    R_ELBOW    = 13, 14
L_WRIST,    R_WRIST    = 15, 16
L_HIP,      R_HIP      = 23, 24

# Tracked joint order: [L_elbow, R_elbow, L_wrist, R_wrist]  (indices 0-3)
TRACK    = [L_ELBOW, R_ELBOW, L_WRIST, R_WRIST]
SHOULDER = [L_SHOULDER, R_SHOULDER, L_SHOULDER, R_SHOULDER]


def angle_cos(a, b, c):
    ba = a - b; bc = c - b
    return float(np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8))


def build_tensor(clip):
    """
    clip: (T, 33, 4)  — world-3D skeleton, columns [x, y, z, vis]
                        x = lateral (left/right), y = vertical (up/down),
                        z = depth (toward camera), shoulder-width normalised.

    Returns float32 tensor of shape (7, T, 4, 3).

    Channel layout
    --------------
    ch0  position xyz          [L_elbow, R_elbow, L_wrist, R_wrist]
    ch1  velocity xyz          frame-to-frame displacement (world 3-D)
    ch2  geometry              bone unit vectors; x-component of elbow rows
                               replaced by elbow flexion angle cosine
    ch3  y-velocity            raw vertical motion per joint (scalar broadcast)
                               (hook: elbow rises; uppercut: wrist rockets up)
    ch4  elbow flexion angle   cos(angle) broadcast to all 4 joint rows
                               (jab ~+1.0 = extended; hook/uppercut ~0.0-0.5)
    ch5  lateral dominance     signed x_vel / ||vel||  — how sideways is the motion?
                               jab  ≈ 0  (forward/z dominant)
                               hook ≈ ±1 (x dominant)
                               uppercut ≈ 0 (y dominant)
    ch6  vertical dominance    signed y_vel / ||vel||  — how upward is the motion?
                               jab      ≈ 0  (z dominant)
                               hook     ≈ 0  (x dominant, stays level)
                               uppercut ≈ +1 (y dominant — wrist drives steeply up)
    """
    T   = clip.shape[0]
    xyz = clip[:, :, :3].astype(np.float32)   # (T, 33, 3)

    tracked = xyz[:, TRACK, :]   # (T, 4, 3)

    # ── ch0: position ────────────────────────────────────────────────────────
    ch0 = tracked.copy()

    # ── ch1: velocity (world 3-D frame-to-frame displacement) ────────────────
    ch1     = np.zeros_like(tracked)
    ch1[1:] = tracked[1:] - tracked[:-1]

    # ── ch2: geometry — bone unit vectors + elbow flexion angle ──────────────
    ch2 = np.zeros_like(tracked)
    for t in range(T):
        f = xyz[t]
        for i in range(4):
            if i < 2:   # elbows: shoulder → elbow
                v = f[TRACK[i]] - f[SHOULDER[i]]
            else:       # wrists: elbow → wrist
                v = f[TRACK[i]] - f[TRACK[i - 2]]
            ch2[t, i] = v / (np.linalg.norm(v) + 1e-8)
        # Overwrite x-component of elbow rows with flexion angle cosine
        ch2[t, 0, 0] = angle_cos(f[L_SHOULDER], f[L_ELBOW], f[L_WRIST])
        ch2[t, 1, 0] = angle_cos(f[R_SHOULDER], f[R_ELBOW], f[R_WRIST])

    # ── ch3: y-velocity — raw vertical motion scalar ─────────────────────────
    # Distinct from ch6 (normalised): raw magnitude matters here — a fast
    # uppercut has large absolute y_vel even before normalising.
    ch3   = np.zeros_like(tracked)
    y_vel = np.zeros((T, 4), dtype=np.float32)
    for i, lm in enumerate(TRACK):
        y_vel[1:, i] = xyz[1:, lm, 1] - xyz[:-1, lm, 1]
    ch3[:, :, :] = y_vel[:, :, np.newaxis]   # broadcast to 3 cols

    # ── ch4: elbow flexion angle, broadcast to all 4 joint rows ──────────────
    ch4 = np.zeros_like(tracked)
    for t in range(T):
        f = xyz[t]
        l_angle = angle_cos(f[L_SHOULDER], f[L_ELBOW], f[L_WRIST])
        r_angle = angle_cos(f[R_SHOULDER], f[R_ELBOW], f[R_WRIST])
        ch4[t, 0, :] = l_angle
        ch4[t, 1, :] = r_angle
        ch4[t, 2, :] = l_angle
        ch4[t, 3, :] = r_angle

    # ── ch5 & ch6: lateral and vertical dominance ────────────────────────────
    # Both use the same denominator (total speed) so they form an
    # interpretable decomposition of motion direction.
    #
    #              jab    hook   uppercut
    #   ch5 (x):   ≈ 0    ≈ ±1    ≈ 0
    #   ch6 (y):   ≈ 0    ≈  0    ≈ +1   ← unique uppercut signature
    #
    # Together they triangulate all 3 classes cleanly.
    eps   = 1e-8
    vel   = ch1                                                   # (T, 4, 3)
    speed = np.linalg.norm(vel, axis=-1, keepdims=True) + eps    # (T, 4, 1)

    lat_dom  = vel[:, :, 0:1] / speed   # x_vel / ||vel||  (T, 4, 1)
    vert_dom = vel[:, :, 1:2] / speed   # y_vel / ||vel||  (T, 4, 1)

    ch5 = np.broadcast_to(lat_dom,  (T, 4, 3)).copy()
    ch6 = np.broadcast_to(vert_dom, (T, 4, 3)).copy()

    return np.stack([ch0, ch1, ch2, ch3, ch4, ch5, ch6], axis=0).astype(np.float32)

assert build_tensor(np.zeros((CLIP_FRAMES, 33, 4), np.float32)).shape == (7, CLIP_FRAMES, 4, 3)
print('Feature shape:', (7, CLIP_FRAMES, 4, 3))

Feature shape: (7, 50, 4, 3)


## 4. Video manifest (not subject IDs)

Class folder names are remapped to canonical output folders: `hook/` → `lead-hook/`, `uppercut/` → `rear-uppercut/`. Every **original video** corresponds to exactly one `.npy` item in the manifest. It checks processed clip availability and ignores unrelated stale `.npy` files rather than silently introducing extra training data. No `subject_mapping.csv` is needed.

In [45]:
def build_manifest(raw_root=INPUT_FOLDER, npy_root=DRIVE_NPY):
    rows = []
    for class_id, label in enumerate(CLASS_NAMES):
        raw_dir = class_folder(raw_root, label)
        if raw_dir is None:
            raise FileNotFoundError(f'No raw folder for {label} (aliases: {CLASS_FOLDERS[label]})')
        videos = source_videos(raw_dir)
        if not videos:
            raise ValueError(f'No raw videos for {label}: {raw_dir}')
        stems = [p.stem.lower() for p in videos]
        if len(set(stems)) != len(stems):
            raise ValueError(f'Nonunique video stems for class {label}: {raw_dir}')
        for video in videos:
            processed = Path(npy_root) / label / (video.stem + '.npy')
            if not processed.is_file():
                raise FileNotFoundError(f'Missing processed clip for {video}: {processed}. Run segment_all().')
            clip = np.load(processed, mmap_mode='r')
            if clip.ndim != 3 or clip.shape[1:] != (33, 4) or clip.shape[0] < 1:
                raise ValueError(f'Bad clip {processed}: {clip.shape}, expected (T,33,4)')
            rows.append({'path': str(processed), 'video_path':str(video),
                         'video_id':f'{label}/{video.stem}',
                         'relative_path':f'{label}/{video.stem}.npy',
                         'label':int(class_id), 'class_name':label})
    df = pd.DataFrame(rows).reset_index(drop=True)
    assert df.video_id.is_unique
    df.to_csv(Path(RUN_DIR)/'video_manifest.csv', index=False)
    print('Video counts:',df.class_name.value_counts().reindex(CLASS_NAMES,fill_value=0).to_dict(), '| Total:',len(df))
    return df

# After segmentation: manifest = build_manifest()

## 5. Training augmentation and datasets

Retain the notebook's existing `v12` (or optional `improved`) spatial/temporal augmentation. Additionally, with probability `WEBCAM_FPS_AUG_PROB=0.5`, degrade the 50-frame clip from 60 FPS to approximately 30 FPS and linearly resample back to 60 FPS before building the seven-channel tensor. This **approximates** the live browser's loss of temporal resolution; it does not reproduce live MediaPipe tracking, irregular timestamps, or how peak detection might shift at 30 FPS. It is applied **only to training data**, never to standard validation/test data. A deterministic webcam-proxy held-out test is also reported separately to illuminate robustness. It should not replace a real webcam evaluation.

The JS app's four-joint speed peak / smoothing / ch5 requirements are recorded in the final deployment checklist. The offline `front_pad` policy for short videos is retained to avoid silently deleting labeled samples; the website currently returns `null` until it has at least 50 resampled frames.


In [46]:
def augment_clip(clip, label_name=None):
    """(T, 33, 4) -> augmented clip. Hooks get stronger augmentation."""
    clip     = clip.copy()
    xyz      = clip[:, :, :3]
    is_hook  = (label_name == 'lead-hook')

    # Mirror left <-> right
    if np.random.random() > 0.5:
        xyz[:, :, 0] = -xyz[:, :, 0]

    # Temporal speed warp
    T     = xyz.shape[0]
    warp  = 0.35 if is_hook else 0.25
    speed = np.random.uniform(1 - warp, 1 + warp)
    src   = np.linspace(0, T - 1, max(int(T * speed), T // 2))
    warped = np.array([xyz[min(int(i), T - 1)] for i in src])
    idx    = np.linspace(0, len(warped) - 1, T)
    xyz    = np.array([warped[min(int(i), len(warped) - 1)] for i in idx])

    # Yaw rotation (larger for hooks to cover more lateral angles)
    max_yaw = 40 if is_hook else 25
    ang = np.random.uniform(-max_yaw, max_yaw) * np.pi / 180
    c, s = np.cos(ang), np.sin(ang)
    R    = np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]], dtype=np.float32)
    xyz  = xyz @ R.T

    # Scale jitter +-12%
    xyz *= np.random.uniform(0.88, 1.12)

    # Gaussian noise
    noise_std = 0.007 if is_hook else 0.005
    xyz += np.random.normal(0, noise_std, xyz.shape).astype(np.float32)

    # Hook-specific: pitch tilt to simulate elbow height variation
    if is_hook and np.random.random() > 0.4:
        pitch = np.random.uniform(-15, 15) * np.pi / 180
        cp, sp = np.cos(pitch), np.sin(pitch)
        Rp = np.array([[1, 0, 0], [0, cp, -sp], [0, sp, cp]], dtype=np.float32)
        xyz = xyz @ Rp.T

    clip[:, :, :3] = xyz
    return clip


# Mirror pairs from MediaPipe's 33 landmark schema (excluding midline landmarks).
_MIRROR_PAIRS = [(1,4),(2,5),(3,6),(7,8),(9,10),(11,12),(13,14),(15,16),
                 (17,18),(19,20),(21,22),(23,24),(25,26),(27,28),(29,30),(31,32)]
_MIRROR_IDX = np.arange(33)
for a,b in _MIRROR_PAIRS:
    _MIRROR_IDX[a],_MIRROR_IDX[b] = b,a

def augment_clip_improved(clip, label_name=None):
    clip = clip.copy()
    xyz = clip[:,:,:3].copy()
    is_hook = HOOK_SPECIFIC_AUG and label_name == 'lead-hook'
    if np.random.rand() < MIRROR_PROB:
        xyz = xyz[:,_MIRROR_IDX,:].copy()
        xyz[:,:,0] *= -1

    # Resample the sequence at a randomized playback speed, retaining output T.
    T = len(xyz)
    speed = np.random.uniform(0.65 if is_hook else 0.75,
                              1.35 if is_hook else 1.25)
    center = (T - 1) / 2
    src = np.clip((np.arange(T)-center) * speed + center, 0, T-1)
    lo, hi = np.floor(src).astype(int), np.ceil(src).astype(int)
    a = (src - lo).astype(np.float32)[:,None,None]
    xyz = xyz[lo]*(1-a) + xyz[hi]*a

    yaw_deg = 40 if is_hook else 25
    ang = np.deg2rad(np.random.uniform(-yaw_deg, yaw_deg))
    c,s = np.cos(ang),np.sin(ang)
    yaw = np.array([[c,0,s],[0,1,0],[-s,0,c]],dtype=np.float32)
    xyz = xyz @ yaw.T
    xyz *= np.random.uniform(0.88,1.12)
    xyz += np.random.normal(0,0.007 if is_hook else 0.005,xyz.shape).astype(np.float32)
    if is_hook and np.random.rand() > 0.4:
        a = np.deg2rad(np.random.uniform(-15,15)); c,s = np.cos(a),np.sin(a)
        pitch = np.array([[1,0,0],[0,c,-s],[0,s,c]],dtype=np.float32)
        xyz = xyz @ pitch.T
    clip[:,:,:3] = xyz
    return clip



def apply_train_augmentation(clip, label_name):
    if AUGMENTATION_MODE == 'none': return clip
    if AUGMENTATION_MODE == 'v12': return augment_clip(clip, label_name)
    if AUGMENTATION_MODE == 'improved': return augment_clip_improved(clip, label_name)
    raise ValueError(f'Unknown AUGMENTATION_MODE: {AUGMENTATION_MODE}')

def prepare_raw_clip(raw):
    raw = np.asarray(raw, dtype=np.float32)
    if raw.ndim != 3 or raw.shape[1:] != (33,4) or len(raw) == 0:
        raise ValueError(f'Expected (T,33,4) skeleton, got {raw.shape}')
    if len(raw) < CLIP_FRAMES:
        padding = np.repeat(raw[:1], CLIP_FRAMES-len(raw), axis=0)
        raw = np.concatenate((padding,raw),axis=0)
    elif len(raw) > CLIP_FRAMES:
        # Extraction guarantees 50; fail on a stale/incorrect upstream clip.
        raise ValueError(f'Expected 50-frame segmented clip, got {len(raw)}. Reprocess with segment_all(force=True).')
    return raw

def simulate_webcam_30_to_60(clip, capture_fps=WEBCAM_CAPTURE_FPS):
    """Approximate sparse webcam sampling and JS-style linear reinterpolation.

    Preserve fixed (50,33,4) shape. The original 50-frame clip spans only
    about 0.83s; sampling its interior is a training-time proxy and does NOT
    exactly simulate re-running the peak detector after 30-FPS acquisition.
    """
    clip = np.asarray(clip, dtype=np.float32)
    if not (0 < capture_fps < TARGET_FPS):
        return clip.copy()
    n = len(clip)
    # Simulate a camera capturing positions at 0, 2, 4, ... for 30->60 FPS.
    src = np.arange(0, n, TARGET_FPS / float(capture_fps), dtype=np.float64)
    if len(src) < 2:
        return clip.copy()
    a = np.floor(src).astype(int)
    b = np.minimum(a + 1, n - 1)
    f = (src - a).astype(np.float32)[:, None, None]
    sparse = clip[a] * (1 - f) + clip[b] * f
    # Match browser's linspace-endpoint temporal resampler, keep EXACTLY 50.
    positions = np.linspace(0, len(sparse) - 1, n, dtype=np.float64)
    lo = np.floor(positions).astype(int)
    hi = np.minimum(lo + 1, len(sparse) - 1)
    frac = (positions - lo).astype(np.float32)[:, None, None]
    return (sparse[lo] * (1 - frac) + sparse[hi] * frac).astype(np.float32)


class BoxingDataset(Dataset):
    def __init__(self, manifest, augment=False, webcam_simulated=False):
        self.frame = manifest.reset_index(drop=True)
        self.augment = augment
        self.webcam_simulated = webcam_simulated
        if augment and webcam_simulated:
            raise ValueError('Do not mix random train augment with deterministic held-out webcam proxy')

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[int(idx)]
        raw = prepare_raw_clip(np.load(row.path))
        if self.augment:
            raw = apply_train_augmentation(raw, row.class_name)
            if WEBCAM_FPS_AUG_PROB and np.random.random() < WEBCAM_FPS_AUG_PROB:
                raw = simulate_webcam_30_to_60(raw)
        elif self.webcam_simulated:
            raw = simulate_webcam_30_to_60(raw)
        feat = build_tensor(raw)
        if not np.isfinite(feat).all():
            raise ValueError(f'Non-finite features in {row.path}')
        return torch.from_numpy(feat), torch.tensor(int(row.label), dtype=torch.long)


## 6. Stratified 5-Fold Cross-Validation (video-level)

Use `sklearn.model_selection.StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)` to allocate **whole videos** to five mutually exclusive test folds. The stratification variable is the **punch label**, not the person's identity or filename. With 40 videos per class, each fold's test set has exactly **8 jab + 8 hook + 8 uppercut = 24 videos**.

For each fold, take the remaining 96 videos and create an **independent stratified validation split** (`VAL_FRACTION=0.15`): **81 train / 15 val / 24 test** for the current balanced 120-video dataset. Test never enters training, early stopping, or checkpoint selection. No augmentation in validation or test. All architectures share the same fixed folds.

`MAX_FOLDS=2` runs only two of the five folds for a smoke test, and is **not** a complete cross-validation result. `MAX_FOLDS=None` evaluates all five folds, covering each video as test exactly once. Videos from the same **person** can appear in both training and test: this evaluates new recordings, not unseen people.


In [47]:
def iter_stratified5_folds(manifest, seed=SEED, max_folds=MAX_FOLDS):
    """Yield 5 stratified VIDEO-level test folds, with separate stratified validation.

    One video = one manifest row. No grouping by individual is performed.
    All videos are held out exactly once as test when max_folds is None.
    """
    manifest = manifest.reset_index(drop=True)
    n = len(manifest)
    labels = manifest.label.to_numpy(dtype=int)
    video_ids = manifest.video_id.astype(str).to_numpy()
    if len(set(video_ids)) != n:
        raise ValueError('video_id must be unique: do not treat clips from one video as independent samples.')
    counts = np.bincount(labels, minlength=N_CLASSES)
    if len(labels) == 0 or any(counts[c] < N_SPLITS for c in range(N_CLASSES)):
        raise ValueError(f'Stratified {N_SPLITS}-fold CV requires at least {N_SPLITS} videos per class; counts={counts.tolist()}')
    if not 0 < VAL_FRACTION < 1:
        raise ValueError(f'VAL_FRACTION must be between 0 and 1, got {VAL_FRACTION}')

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    split_pairs = list(skf.split(np.zeros(n), labels))
    if max_folds is not None and not (1 <= int(max_folds) <= N_SPLITS):
        raise ValueError(f'MAX_FOLDS must be 1..{N_SPLITS} or None; got {max_folds}')
    chosen = range(N_SPLITS if max_folds is None else int(max_folds))
    for fold_id in chosen:
        train_val_idx, test_idx = split_pairs[fold_id]
        try:
            train_idx, val_idx = train_test_split(
                train_val_idx,
                test_size=VAL_FRACTION,
                stratify=labels[train_val_idx],
                random_state=seed + 1000 + fold_id,
                shuffle=True,
            )
        except ValueError as exc:
            raise ValueError(
                f'Fold {fold_id}: not enough class examples for a separate stratified validation set. '
                f'Increase dataset size or VAL_FRACTION; original error: {exc}'
            ) from exc
        train_idx = np.sort(np.asarray(train_idx, dtype=int))
        val_idx = np.sort(np.asarray(val_idx, dtype=int))
        test_idx = np.sort(np.asarray(test_idx, dtype=int))
        train_set, val_set, test_set = set(train_idx), set(val_idx), set(test_idx)
        if train_set & val_set or train_set & test_set or val_set & test_set:
            raise AssertionError(f'Fold {fold_id}: train/val/test overlap.')
        if (train_set | val_set | test_set) != set(range(n)):
            raise AssertionError(f'Fold {fold_id}: missing videos.')
        if any(set(labels[idx]) != set(range(N_CLASSES)) for idx in (train_idx, val_idx, test_idx)):
            raise ValueError(f'Fold {fold_id}: one or more classes are missing in train/val/test.')
        yield {
            'fold': fold_id, 'train': train_idx, 'val': val_idx, 'test': test_idx,
            'test_video_ids': video_ids[test_idx].tolist(), 'n_test': len(test_idx),
        }


def check_folds(manifest, max_folds=MAX_FOLDS):
    """Print counts and audit uniqueness, stratification, and video overlap."""
    folds = list(iter_stratified5_folds(manifest, max_folds=max_folds))
    test_indices = [int(i) for f in folds for i in f['test']]
    assert len(test_indices) == len(set(test_indices)), 'A video appears in test more than once.'
    for f in folds:
        test_classes = manifest.iloc[f['test']].class_name.value_counts().reindex(CLASS_NAMES, fill_value=0).to_dict()
        print(f"Fold {f['fold']:03d} | train={len(f['train']):3d} "
              f"val={len(f['val']):3d} test={len(f['test']):3d} | "
              f"test_classes={test_classes}")
    print(f'Test coverage: {len(test_indices)}/{len(manifest)} unique videos across {len(folds)} folds.')
    if max_folds is None:
        assert len(folds) == N_SPLITS and sorted(test_indices) == list(range(len(manifest)))
    return folds

# After build_manifest(): check_folds(manifest)


## 7. Large model zoo — exactly five architectures, one feature contract

Every model receives `(B,7,50,4,3)` and produces `(B,3)` logits. All models are **randomly initialized** (no official pretrained weights) and have a `*_large` name because they are substantially larger than this notebook's earlier small models.

| Model key | Backbone (task-adapted) | Main design |
|---|---|---|
| `dinat_large` | 8 blocks, 192 channels, 8 heads | Joint attention plus alternating local and dilated temporal neighborhood attention (dilation `1,2,1,4,1,8,1,2`), FFNs and LayerScale |
| `stgcn_large` | 8 blocks, channels 96–256 | 3 graph partitions (self, same-arm, contralateral), learned edge importance and residual temporal convolutions |
| `semgcn_large` | 8 blocks, channels 128–256 | Learnable softmax semantic adjacency over **all four nodes** with four heads, residual dilated temporal convolutions |
| `motionbert_large` | 6 modules, 160 channels, 8 heads | MotionBERT/DSTformer-inspired spatial→temporal and temporal→spatial streams with per-token adaptive fusion |
| `dastformer_large` | 6 modules, 144 channels, 6 heads | DASTFormer-inspired spatial→temporal, temporal→spatial, and parallel branches with attention-adaptive (AtA-like) and pure-adaptive (PuA-like) fusion |

**Architectural attribution (adaptations, not paper-faithful reproductions):** DiNAT: [Hassani & Shi, DiNAT](https://github.com/SHI-Labs/Neighborhood-Attention-Transformer/blob/main/DiNAT.md); ST-GCN: [Yan et al., AAAI 2018](https://github.com/LevelCA/st-gcn); SemGCN: [Zhao et al., CVPR 2019](https://openaccess.thecvf.com/content_CVPR_2019/html/Zhao_Semantic_Graph_Convolutional_Networks_for_3D_Human_Pose_Regression_CVPR_2019_paper.html); MotionBERT: [Zhu et al., ICCV 2023](https://openaccess.thecvf.com/content/ICCV2023/papers/Zhu_MotionBERT_A_Unified_Perspective_on_Learning_Human_Motion_Representations_ICCV_2023_paper.pdf); DASTFormer: [Learning Temporal–Spatial Contextual Adaptation, 2024](https://pmc.ncbi.nlm.nih.gov/articles/PMC11244605/).

**Fair-comparison warning:** `BATCH_SIZE=8`, learning rate `2e-4`, and weight decay `1e-2` are shared across all five for this large-model comparison. This is a usable baseline, not per-architecture hyperparameter optimization. The custom graph is only **four arm nodes**, not a 17/25/33-joint human skeleton; spatial graph expressiveness is necessarily restricted.


In [48]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# All five architectures are adapted for the SAME four-joint feature tensor:
# (B, 7, 50, 4, 3) -> (B, 3) logits. These are training-from-scratch,
# task-adapted models, NOT official paper checkpoints or pretrained backbones.
NODES = 4
NODE_FEATURES = N_CHANNELS * 3  # 21 features at each arm joint


def _check_input(x):
    if x.ndim != 5 or x.shape[1:] != (N_CHANNELS, CLIP_FRAMES, NODES, 3):
        raise ValueError(f'Expected (B,{N_CHANNELS},{CLIP_FRAMES},4,3), got {tuple(x.shape)}')


def _joint_tokens(x):
    _check_input(x)
    b, c, t, j, d = x.shape
    return x.permute(0, 2, 3, 1, 4).reshape(b, t, j, c * d)


def _conv_tokens(x):
    _check_input(x)
    b, c, t, j, d = x.shape
    return x.permute(0, 1, 4, 2, 3).reshape(b, c * d, t, j)


class PoolClassifier(nn.Module):
    """Preserve mean and strong peak-motion activations before classification."""
    def __init__(self, dim, n_classes, dropout=0.35):
        super().__init__()
        self.head = nn.Sequential(
            nn.LayerNorm(2 * dim), nn.Dropout(dropout),
            nn.Linear(2 * dim, dim), nn.GELU(),
            nn.Dropout(dropout / 2), nn.Linear(dim, n_classes),
        )

    def forward(self, x):  # (B,T,J,D)
        return self.head(torch.cat([x.mean(dim=(1, 2)), x.amax(dim=(1, 2))], dim=-1))


# Normalized graph partitions for four arm joints: self, same-arm chain,
# and cross-arm similarity. Joint order L/R elbow, L/R wrist.
def _graph_partitions():
    eye = torch.eye(NODES)
    same_arm = torch.zeros(NODES, NODES)
    cross_arm = torch.zeros(NODES, NODES)
    for a, b in ((0, 2), (1, 3)):
        same_arm[a, b] = same_arm[b, a] = 1.
    for a, b in ((0, 1), (2, 3)):
        cross_arm[a, b] = cross_arm[b, a] = 1.
    def norm(a):
        degree = a.sum(-1).clamp_min(1.)
        return degree.rsqrt()[:, None] * a * degree.rsqrt()[None, :]
    return torch.stack([norm(eye), norm(same_arm), norm(cross_arm)])


class LargeSTGCNBlock(nn.Module):
    """Partitioned spatial graph convolution + temporal CNN + residual."""
    def __init__(self, cin, cout, stride=1, dropout=0.15):
        super().__init__()
        self.cout = cout
        self.graph_proj = nn.Conv2d(cin, cout * 3, 1, bias=False)
        self.register_buffer('adj', _graph_partitions())
        self.edge_scale = nn.Parameter(torch.ones(3, NODES, NODES))
        self.graph_norm = nn.BatchNorm2d(cout)
        self.temporal = nn.Sequential(
            nn.Conv2d(cout, cout, (7, 1), stride=(stride, 1),
                      padding=(3, 0), bias=False),
            nn.BatchNorm2d(cout), nn.Dropout(dropout),
        )
        self.skip = (nn.Identity() if cin == cout and stride == 1 else
                     nn.Sequential(nn.Conv2d(cin, cout, 1, stride=(stride, 1), bias=False),
                                   nn.BatchNorm2d(cout)))

    def forward(self, x):
        b, _, t, j = x.shape
        projected = self.graph_proj(x).reshape(b, 3, self.cout, t, j)
        mixed = torch.einsum('bkctj,kij->bcti', projected, self.adj * self.edge_scale)
        return F.gelu(self.temporal(F.gelu(self.graph_norm(mixed))) + self.skip(x))


class LargeSTGCN(nn.Module):
    """Eight graph-temporal blocks, three adjacency partitions, 96–256 channels."""
    def __init__(self, n_classes=N_CLASSES):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(NODE_FEATURES, 96, 1, bias=False),
                                  nn.BatchNorm2d(96), nn.GELU())
        channels = (96, 96, 128, 128, 192, 192, 256, 256)
        blocks = []
        cin = 96
        for i, cout in enumerate(channels):
            blocks.append(LargeSTGCNBlock(cin, cout, stride=2 if i in (3, 5) else 1,
                                          dropout=0.10 + 0.015 * i))
            cin = cout
        self.blocks = nn.Sequential(*blocks)
        self.norm = nn.BatchNorm2d(cin)
        self.head = PoolClassifier(cin, n_classes)

    def forward(self, x):
        x = self.norm(self.blocks(self.stem(_conv_tokens(x))))
        return self.head(x.permute(0, 2, 3, 1))


class SemanticGraphConv(nn.Module):
    """Learn a softmax-normalized semantic adjacency separately per attention head.

    Unlike fixed ST-GCN adjacency, every joint pair can have a learned weight.
    The physical graph supplies an initialization bias, not a hard restriction.
    """
    def __init__(self, cin, cout, heads=4):
        super().__init__()
        prior = _graph_partitions().sum(0) > 0
        bias = torch.where(prior, torch.tensor(1.5), torch.tensor(-1.5))
        self.semantic_logits = nn.Parameter(bias.unsqueeze(0).expand(heads, -1, -1).clone())
        self.project = nn.Conv2d(cin * heads, cout, 1, bias=False)
        self.norm = nn.BatchNorm2d(cout)

    def forward(self, x):  # (B,C,T,J)
        adjacency = self.semantic_logits.softmax(dim=-1)
        x = torch.einsum('bctj,hij->bhcti', x, adjacency)
        b, h, c, t, j = x.shape
        return self.norm(self.project(x.reshape(b, h * c, t, j)))


class LargeSemGCNBlock(nn.Module):
    def __init__(self, cin, cout, stride=1, dilation=1, dropout=0.15):
        super().__init__()
        self.semantic = SemanticGraphConv(cin, cout, heads=4)
        self.temporal = nn.Sequential(
            nn.Conv2d(cout, cout, (5, 1), stride=(stride, 1),
                      padding=(2 * dilation, 0), dilation=(dilation, 1), bias=False),
            nn.BatchNorm2d(cout), nn.Dropout(dropout),
        )
        self.skip = (nn.Identity() if cin == cout and stride == 1 else
                     nn.Sequential(nn.Conv2d(cin, cout, 1, stride=(stride, 1), bias=False),
                                   nn.BatchNorm2d(cout)))

    def forward(self, x):
        return F.gelu(self.temporal(F.gelu(self.semantic(x))) + self.skip(x))


class LargeSemGCN(nn.Module):
    """Eight residual semantic-graph blocks and dilated temporal convolutions."""
    def __init__(self, n_classes=N_CLASSES):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(NODE_FEATURES, 128, 1, bias=False),
                                  nn.BatchNorm2d(128), nn.GELU())
        channels = (128, 128, 160, 160, 192, 192, 256, 256)
        cin, blocks = 128, []
        for i, cout in enumerate(channels):
            blocks.append(LargeSemGCNBlock(cin, cout, stride=2 if i in (3, 5) else 1,
                                           dilation=(1, 2, 4, 1)[i % 4],
                                           dropout=0.10 + 0.015 * i))
            cin = cout
        self.blocks = nn.Sequential(*blocks)
        self.norm = nn.BatchNorm2d(cin)
        self.head = PoolClassifier(cin, n_classes)

    def forward(self, x):
        x = self.norm(self.blocks(self.stem(_conv_tokens(x))))
        return self.head(x.permute(0, 2, 3, 1))


class MultiHeadAttention(nn.Module):
    """Export-friendly MHSA used for the 4-joint spatial / 50-frame temporal axes."""
    def __init__(self, dim, heads=8, attn_dropout=0.05):
        super().__init__()
        assert dim % heads == 0
        self.heads = heads
        self.scale = (dim // heads) ** -0.5
        self.qkv = nn.Linear(dim, 3 * dim)
        self.out = nn.Linear(dim, dim)
        self.drop = nn.Dropout(attn_dropout)

    def forward(self, x):  # (N, L, D)
        n, length, dim = x.shape
        qkv = self.qkv(x).reshape(n, length, 3, self.heads, dim // self.heads)
        q, k, v = qkv.unbind(2)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        weights = self.drop(torch.matmul(q, k.transpose(-2, -1)).mul(self.scale).softmax(dim=-1))
        result = torch.matmul(weights, v).transpose(1, 2).reshape(n, length, dim)
        return self.out(result)


class AxialTransformerUnit(nn.Module):
    """Pre-norm spatial OR temporal MHSA + MLP, each with LayerScale residuals."""
    def __init__(self, dim, axis, heads=8, mlp_ratio=3, dropout=0.10):
        super().__init__()
        assert axis in ('spatial', 'temporal')
        self.axis = axis
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MultiHeadAttention(dim, heads, attn_dropout=dropout / 2)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(nn.Linear(dim, int(dim * mlp_ratio)), nn.GELU(),
                                 nn.Dropout(dropout), nn.Linear(int(dim * mlp_ratio), dim))
        self.drop = nn.Dropout(dropout)
        self.attn_scale = nn.Parameter(torch.full((dim,), 0.1))
        self.ffn_scale = nn.Parameter(torch.full((dim,), 0.1))

    def forward(self, x):  # (B,T,J,D)
        b, t, j, d = x.shape
        z = self.norm1(x)
        if self.axis == 'spatial':
            z = self.attn(z.reshape(b * t, j, d)).reshape(b, t, j, d)
        else:
            z = z.permute(0, 2, 1, 3).reshape(b * j, t, d)
            z = self.attn(z).reshape(b, j, t, d).permute(0, 2, 1, 3)
        x = x + self.drop(z) * self.attn_scale
        return x + self.drop(self.ffn(self.norm2(x))) * self.ffn_scale


class TemporalDilatedNeighborhoodAttention(nn.Module):
    """Local temporal self-attention at dilated offsets, applied joint-wise."""
    def __init__(self, dim=192, heads=8, kernel=7, dilation=1, attn_dropout=0.05):
        super().__init__()
        assert dim % heads == 0 and kernel % 2 == 1
        self.heads = heads
        self.kernel = kernel
        self.dilation = dilation
        self.scale = (dim // heads) ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.out = nn.Linear(dim, dim)
        self.drop = nn.Dropout(attn_dropout)

    def forward(self, x):  # (B,T,J,D)
        b, t, j, dim = x.shape
        z = x.permute(0, 2, 1, 3).reshape(b * j, t, dim)
        q, k, v = self.qkv(z).chunk(3, dim=-1)
        dh = dim // self.heads
        q = q.reshape(b * j, t, self.heads, dh)
        k = k.reshape(b * j, t, self.heads, dh)
        v = v.reshape(b * j, t, self.heads, dh)
        offset = torch.arange(-(self.kernel // 2), self.kernel // 2 + 1,
                              device=x.device) * self.dilation
        index = torch.arange(t, device=x.device)[:, None] + offset[None, :]
        valid = (index >= 0) & (index < t)
        index = index.clamp(0, t - 1)
        kn, vn = k[:, index], v[:, index]  # (B*J,T,K,H,Dh)
        scores = (q[:, :, None] * kn).sum(-1) * self.scale
        scores = scores.masked_fill(~valid[None, :, :, None], -1.e4)
        weights = self.drop(scores.softmax(dim=2))
        result = (weights[:, :, :, :, None] * vn).sum(dim=2).reshape(b * j, t, dim)
        return self.out(result).reshape(b, j, t, dim).permute(0, 2, 1, 3)


class LargeDiNATBlock(nn.Module):
    """Spatial joint MHSA, temporal dilated neighborhood attention, and FFN."""
    def __init__(self, dim=192, heads=8, dilation=1, dropout=0.10):
        super().__init__()
        self.joint = AxialTransformerUnit(dim, 'spatial', heads, mlp_ratio=3, dropout=dropout)
        self.norm1 = nn.LayerNorm(dim)
        self.temporal = TemporalDilatedNeighborhoodAttention(dim, heads, kernel=7,
                                                               dilation=dilation, attn_dropout=dropout / 2)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(4 * dim, dim))
        self.drop = nn.Dropout(dropout)
        self.scale_time = nn.Parameter(torch.full((dim,), 0.1))
        self.scale_ffn = nn.Parameter(torch.full((dim,), 0.1))

    def forward(self, x):
        x = self.joint(x)
        x = x + self.drop(self.temporal(self.norm1(x))) * self.scale_time
        return x + self.drop(self.ffn(self.norm2(x))) * self.scale_ffn


class LargeDiNAT(nn.Module):
    """Eight blocks, width 192; alternating local/dilated temporal attention."""
    def __init__(self, n_classes=N_CLASSES):
        super().__init__()
        dim = 192
        self.embed = nn.Sequential(nn.Linear(NODE_FEATURES, dim), nn.LayerNorm(dim))
        self.joint_pos = nn.Parameter(torch.randn(1, 1, NODES, dim) * 0.02)
        self.time_pos = nn.Parameter(torch.randn(1, CLIP_FRAMES, 1, dim) * 0.02)
        self.blocks = nn.Sequential(*[
            LargeDiNATBlock(dim, heads=8, dilation=d, dropout=0.10)
            for d in (1, 2, 1, 4, 1, 8, 1, 2)
        ])
        self.norm = nn.LayerNorm(dim)
        self.head = PoolClassifier(dim, n_classes)

    def forward(self, x):
        z = self.embed(_joint_tokens(x)) + self.joint_pos + self.time_pos
        return self.head(self.norm(self.blocks(z)))


class MotionBERTDualStreamBlock(nn.Module):
    """DSTformer-inspired dual orderings: spatial->temporal, temporal->spatial.

    Adaptive per-token fusion preserves the key MotionBERT architectural idea.
    Trained from scratch for classification (not the original pretrained lifter).
    """
    def __init__(self, dim=160, heads=8, dropout=0.10):
        super().__init__()
        self.st = nn.Sequential(
            AxialTransformerUnit(dim, 'spatial', heads, dropout=dropout),
            AxialTransformerUnit(dim, 'temporal', heads, dropout=dropout),
        )
        self.ts = nn.Sequential(
            AxialTransformerUnit(dim, 'temporal', heads, dropout=dropout),
            AxialTransformerUnit(dim, 'spatial', heads, dropout=dropout),
        )
        self.fusion = nn.Sequential(nn.LayerNorm(dim * 2), nn.Linear(dim * 2, 2))
        self.fusion_scale = nn.Parameter(torch.tensor(0.5))

    def forward(self, x):
        a, b = self.st(x), self.ts(x)
        weights = self.fusion(torch.cat([a, b], dim=-1)).softmax(dim=-1)
        merged = weights[..., 0:1] * a + weights[..., 1:2] * b
        return x + self.fusion_scale * (merged - x)


class LargeMotionBERT(nn.Module):
    """Six dual-stream DSTformer-inspired blocks; 160 channels and 8 heads."""
    def __init__(self, n_classes=N_CLASSES):
        super().__init__()
        dim = 160
        self.embed = nn.Sequential(nn.Linear(NODE_FEATURES, dim), nn.LayerNorm(dim))
        self.joint_pos = nn.Parameter(torch.randn(1, 1, NODES, dim) * 0.02)
        self.time_pos = nn.Parameter(torch.randn(1, CLIP_FRAMES, 1, dim) * 0.02)
        self.blocks = nn.Sequential(*[MotionBERTDualStreamBlock(dim, heads=8) for _ in range(6)])
        self.norm = nn.LayerNorm(dim)
        self.head = PoolClassifier(dim, n_classes)

    def forward(self, x):
        z = self.embed(_joint_tokens(x)) + self.joint_pos + self.time_pos
        return self.head(self.norm(self.blocks(z)))


class DASTFormerTripleBranchBlock(nn.Module):
    """DASTFormer-inspired three-branch spatial-temporal adaptive fusion.

    Two serial orderings and one parallel branch are combined using AtA-like
    per-token attention and PuA-like learned global mixture. Not a strict paper
    reproduction (the original predicts 3D human poses, not punch classes).
    """
    def __init__(self, dim=144, heads=6, dropout=0.10):
        super().__init__()
        self.st = nn.Sequential(AxialTransformerUnit(dim, 'spatial', heads, dropout=dropout),
                                AxialTransformerUnit(dim, 'temporal', heads, dropout=dropout))
        self.ts = nn.Sequential(AxialTransformerUnit(dim, 'temporal', heads, dropout=dropout),
                                AxialTransformerUnit(dim, 'spatial', heads, dropout=dropout))
        self.parallel_s = AxialTransformerUnit(dim, 'spatial', heads, dropout=dropout)
        self.parallel_t = AxialTransformerUnit(dim, 'temporal', heads, dropout=dropout)
        self.attn_adaptive = nn.Sequential(nn.LayerNorm(dim * 3), nn.Linear(dim * 3, 3))
        self.pure_adaptive = nn.Parameter(torch.zeros(3))
        self.mode_gate = nn.Sequential(nn.LayerNorm(dim * 3), nn.Linear(dim * 3, 1))
        self.fusion_scale = nn.Parameter(torch.tensor(0.5))

    def forward(self, x):
        a, b = self.st(x), self.ts(x)
        c = 0.5 * (self.parallel_s(x) + self.parallel_t(x))
        fused_input = torch.cat([a, b, c], dim=-1)
        ata = self.attn_adaptive(fused_input).softmax(dim=-1)
        attn_mix = ata[..., 0:1] * a + ata[..., 1:2] * b + ata[..., 2:3] * c
        pua = self.pure_adaptive.softmax(dim=0)
        pure_mix = pua[0] * a + pua[1] * b + pua[2] * c
        gate = torch.sigmoid(self.mode_gate(fused_input))
        merged = gate * attn_mix + (1.0 - gate) * pure_mix
        return x + self.fusion_scale * (merged - x)


class LargeDASTFormer(nn.Module):
    """Six three-branch DASTFormer-inspired blocks; 144 channels, six heads."""
    def __init__(self, n_classes=N_CLASSES):
        super().__init__()
        dim = 144
        self.embed = nn.Sequential(nn.Linear(NODE_FEATURES, dim), nn.LayerNorm(dim))
        self.joint_pos = nn.Parameter(torch.randn(1, 1, NODES, dim) * 0.02)
        self.time_pos = nn.Parameter(torch.randn(1, CLIP_FRAMES, 1, dim) * 0.02)
        self.blocks = nn.Sequential(*[DASTFormerTripleBranchBlock(dim, heads=6) for _ in range(6)])
        self.norm = nn.LayerNorm(dim)
        self.head = PoolClassifier(dim, n_classes)

    def forward(self, x):
        z = self.embed(_joint_tokens(x)) + self.joint_pos + self.time_pos
        return self.head(self.norm(self.blocks(z)))


# Exactly five requested models: no tiny baselines, CNNs, TCNs or generic transformers.
MODEL_ZOO = {
    'dinat_large': LargeDiNAT,
    'stgcn_large': LargeSTGCN,
    'semgcn_large': LargeSemGCN,
    'motionbert_large': LargeMotionBERT,
    'dastformer_large': LargeDASTFormer,
}
assert set(MODEL_NAMES) == set(MODEL_ZOO)


def create_model(name):
    if name not in MODEL_ZOO:
        raise ValueError(f'Unknown model {name!r}; choices: {list(MODEL_ZOO)}')
    return MODEL_ZOO[name](n_classes=N_CLASSES)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def print_comparison_footprints():
    print('Architecture | trainable parameters | FP32 parameters (MiB; excludes activations)')
    for name in MODEL_NAMES:
        model = create_model(name)
        p = count_parameters(model)
        print(f'{name:20s} | {p:>12,} | {p * 4 / 1048576:>8.2f}')
    print('For real ONNX file sizes, export a trained checkpoint with export_onnx().')


print_comparison_footprints()


Architecture | trainable parameters | FP32 parameters (MiB; excludes activations)
dinat_large          |    6,623,235 |    25.27
stgcn_large          |    2,744,355 |    10.47
semgcn_large         |    2,796,035 |    10.67
motionbert_large     |    6,270,741 |    23.92
dastformer_large     |    7,612,467 |    29.04
For real ONNX file sizes, export a trained checkpoint with export_onnx().


## 8. Experiment training / held-out predictions

For each of the **five** large architectures, each fold uses only training videos for gradient updates and validation videos for best-epoch selection via cross-entropy loss (macro-F1 tie-breaker). Held-out test videos are used once **after** checkpoint selection. Twenty-five fold training jobs run when `RUN_EXPERIMENTS=True` and `MAX_FOLDS=None`.

Because the models are larger and the dataset contains approximately 120 videos, watch for train/validation divergence and GPU memory use. Start with `MAX_FOLDS=1` as a quick sanity run, then restore all five folds for the final comparison. Cross-validation checkpoints are **not** final deployment models: run the optional full-data refit only after comparing all five complete held-out results.


In [49]:
def loaders_for_fold(manifest, f):
    clean_ds=BoxingDataset(manifest,augment=False)
    train_ds=BoxingDataset(manifest,augment=AUGMENT_TRAIN)
    ids = f['train']
    labels=manifest.iloc[ids].label.to_numpy()
    if SAMPLER_MODE=='balanced':
        counts=np.bincount(labels,minlength=N_CLASSES)
        weights=1.0 / np.maximum(counts[labels],1)
    elif SAMPLER_MODE=='hook_x3':
        weights=np.where(labels==CLASS_NAMES.index('lead-hook'),3.,1.)
    elif SAMPLER_MODE=='shuffle':
        weights=None
    else:
        raise ValueError(SAMPLER_MODE)
    sampler=(WeightedRandomSampler(torch.as_tensor(weights,dtype=torch.double),
                num_samples=len(ids),replacement=True) if weights is not None else None)
    train=DataLoader(Subset(train_ds,ids.tolist()),batch_size=BATCH_SIZE,
                     shuffle=sampler is None,sampler=sampler,num_workers=NUM_WORKERS,
                     pin_memory=torch.cuda.is_available())
    def val_loader(indices):
        return DataLoader(Subset(clean_ds,indices.tolist()),batch_size=BATCH_SIZE,
                          shuffle=False,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available())
    return train,val_loader(f['val']),val_loader(f['test'])

def run_one_epoch(model,loader,device,optimizer=None):
    training=optimizer is not None
    model.train(training)
    criterion=nn.CrossEntropyLoss(label_smoothing=0.05 if training else 0.0)
    total_loss,n=0.,0
    truth,preds=[],[]
    with torch.set_grad_enabled(training):
        for X,y in loader:
            X=X.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
            if training: optimizer.zero_grad(set_to_none=True)
            logits=model(X)
            loss=criterion(logits,y)
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.)
                optimizer.step()
            total_loss+=loss.item()*len(y);n+=len(y)
            truth.extend(y.detach().cpu().tolist())
            preds.extend(logits.argmax(1).detach().cpu().tolist())
    if not n: raise ValueError('Empty DataLoader in epoch')
    return {'loss':total_loss/n,'acc':accuracy_score(truth,preds),
            'macro_f1':f1_score(truth,preds,labels=list(range(N_CLASSES)),
                               average='macro',zero_division=0)}

@torch.no_grad()
def collect_predictions(model,loader,device):
    model.eval()
    labels,preds,probs=[],[],[]
    for X,y in loader:
        out=model(X.to(device)).softmax(1).cpu().numpy()
        labels.extend(y.numpy().tolist())
        preds.extend(out.argmax(1).tolist())
        probs.extend(out.tolist())
    return np.asarray(labels),np.asarray(preds),np.asarray(probs)



def train_one_fold(manifest, f, model_name, device):
    seed_everything(SEED + f['fold'])
    train_loader, val_loader, test_loader = loaders_for_fold(manifest, f)
    model = create_model(model_name).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    best_loss=float('inf'); best_f1=-1.; wait=0; best_epoch=None
    model_dir = os.path.join(RUN_DIR, model_name)
    os.makedirs(model_dir, exist_ok=True)
    ckpt = os.path.join(model_dir, f"fold_{f['fold']:03d}_stratified5.pt")
    for epoch in range(1, EPOCHS+1):
        tr = run_one_epoch(model, train_loader, device, optimizer)
        val = run_one_epoch(model, val_loader, device)
        scheduler.step()
        better = (val['loss'] < best_loss-1e-5 or
                  (abs(val['loss']-best_loss)<=1e-5 and val['macro_f1']>best_f1))
        if better:
            best_loss, best_f1 = val['loss'], val['macro_f1']
            best_epoch=epoch; wait=0
            torch.save({'state_dict': model.state_dict(), 'model_name': model_name,
                        'classes': CLASS_NAMES, 'clip_frames': CLIP_FRAMES, 'fps':TARGET_FPS, 'n_channels':N_CHANNELS, 'augmentation_mode':AUGMENTATION_MODE, 'pose_variant': POSE_VARIANT,
                        'fold': f['fold'], 'test_video_ids': f['test_video_ids'],
                        'validation_video_ids': manifest.iloc[f['val']].video_id.tolist(),
                        'split_strategy': 'stratified_5_fold_video_level_60fps_peak_v25',
                        'preprocessing_contract': PREPROCESS_CONTRACT,
                        'webcam_fps_aug_prob': WEBCAM_FPS_AUG_PROB,
                        'webcam_capture_fps': WEBCAM_CAPTURE_FPS,
                        'epoch': epoch}, ckpt)
        else:
            wait += 1
        if epoch==1 or epoch%10==0 or better:
            print(f"{model_name} fold={f['fold']} ep={epoch:03d} "
                  f"train_acc={tr['acc']:.3f} val_f1={val['macro_f1']:.3f} "
                  f"val_loss={val['loss']:.3f}{' *' if better else ''}")
        if wait >= PATIENCE:
            break
    saved = torch.load(ckpt, map_location=device, weights_only=True)
    model.load_state_dict(saved['state_dict'])
    actual, pred, prob = collect_predictions(model, test_loader, device)
    # A SECOND held-out report with a deterministic sparse-30-FPS proxy.
    # This is diagnostic only: no real webcam or end-to-end JS tracking here.
    webcam_ds = BoxingDataset(manifest, augment=False, webcam_simulated=True)
    webcam_loader = DataLoader(Subset(webcam_ds, f['test'].tolist()),
                               batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    actual_web, pred_web, prob_web = collect_predictions(model, webcam_loader, device)
    assert np.array_equal(actual, actual_web)
    records=[]
    webcam_records=[]
    for i, index in enumerate(f['test']):
        row = manifest.iloc[int(index)]
        rec={'model':model_name, 'fold': f['fold'], 'video_id':row.video_id,
             'relative_path':row.relative_path, 'path':row.path,
             'class_name':row.class_name,
             'true':int(actual[i]), 'pred':int(pred[i]),
             'confidence':float(prob[i].max()),'best_epoch':best_epoch}
        for k,cls in enumerate(CLASS_NAMES):
            rec[f'p_{cls}'] = float(prob[i,k])
        records.append(rec)
        wc = dict(rec)
        wc['pred'] = int(pred_web[i])
        wc['confidence'] = float(prob_web[i].max())
        for k, cls in enumerate(CLASS_NAMES):
            wc[f'p_{cls}'] = float(prob_web[i, k])
        webcam_records.append(wc)
    return records, webcam_records, ckpt


def evaluate_stratified5(manifest, model_names=MODELS_TO_RUN, max_folds=MAX_FOLDS):
    """Evaluate fixed stratified 5-fold test sets; never use test data for early stopping."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    folds = list(iter_stratified5_folds(manifest, max_folds=max_folds))
    print(f'Running {len(folds)} folds x {len(model_names)} model(s); '
          f'{sum(len(f["test"]) for f in folds)}/{len(manifest)} test videos.')
    all_records=[]; webcam_records=[]; checkpoints=[]
    for model_name in model_names:
        print('\n'+'='*75+'\nMODEL:',model_name)
        for f in folds:
            records, simulated_records, ckpt = train_one_fold(manifest, f, model_name, device)
            all_records.extend(records)
            webcam_records.extend(simulated_records)
            checkpoints.append({'model':model_name, 'fold':f['fold'],
                                'num_test_videos':len(f['test']), 'checkpoint':ckpt})
            pd.DataFrame(all_records).to_csv(
                os.path.join(RUN_DIR,'stratified5_predictions_partial.csv'),index=False)
    predictions=pd.DataFrame(all_records)
    predictions.to_csv(os.path.join(RUN_DIR, 'stratified5_predictions.csv'),index=False)
    pd.DataFrame(webcam_records).to_csv(
        os.path.join(RUN_DIR, 'stratified5_webcam_proxy_predictions.csv'), index=False)
    pd.DataFrame(checkpoints).to_csv(os.path.join(RUN_DIR, 'stratified5_checkpoints.csv'),index=False)
    print('Saved predictions/checkpoints to', RUN_DIR)
    return predictions

# manifest = build_manifest()
# check_folds(manifest)
# predictions = evaluate_stratified5(manifest, MODELS_TO_RUN)  # five large candidates


## 9. Results, confusion matrix, per-video mistakes

In [50]:
def summarize_stratified5(predictions, show_plots=True, filename='stratified5_summary.csv'):
    if predictions.empty:
        raise ValueError('No predictions yet. Run evaluate_stratified5().')
    summary=[]
    for name, frame in predictions.groupby('model', sort=False):
        yt=frame.true.to_numpy(dtype=int); yp=frame.pred.to_numpy(dtype=int)
        per_fold = frame.groupby('fold', sort=True).apply(
            lambda g: pd.Series({'accuracy': accuracy_score(g.true,g.pred),
              'macro_f1': f1_score(g.true,g.pred,labels=list(range(N_CLASSES)),
                                   average='macro',zero_division=0),
              'test_videos':len(g)}), include_groups=False)
        row={'model':name, 'tested_videos':len(frame), 'folds':len(per_fold),
             'pooled_accuracy':accuracy_score(yt,yp),
             'pooled_macro_f1':f1_score(yt,yp,labels=list(range(N_CLASSES)),
                                      average='macro',zero_division=0),
             'mean_fold_accuracy':per_fold.accuracy.mean(),
             'mean_fold_macro_f1':per_fold.macro_f1.mean()}
        summary.append(row)
        print('\n',name,'\n',pd.DataFrame([row]).to_string(index=False))
        print('\nPer-fold:\n',per_fold.to_string())
        print('\nClass breakdown:\n',classification_report(
            yt,yp,labels=list(range(N_CLASSES)),target_names=CLASS_NAMES,
            digits=3,zero_division=0))
        if show_plots:
            disp=ConfusionMatrixDisplay(confusion_matrix(
                yt,yp,labels=list(range(N_CLASSES))),display_labels=CLASS_NAMES)
            disp.plot(cmap='Blues',values_format='d',xticks_rotation=20)
            plt.title(f'{name}: video-held-out pooled predictions')
            plt.tight_layout();plt.show()
    table=pd.DataFrame(summary)
    table.to_csv(os.path.join(RUN_DIR,filename),index=False)
    return table

# summary = summarize_stratified5(predictions)
# errors = predictions[predictions.true != predictions.pred]

## 10. Sanity checks: feature contract, five large model gradients, and stratified fold coverage

With no Google Drive videos, the checks generate one synthetic `(1,7,50,4,3)` tensor and verify **all five** model forward passes, finite logits, backward gradients, the 4-joint speed peak, the webcam FPS proxy, and the five video-level stratified folds. This validates shapes and differentiability, **not** numerical correctness of pretrained weights, real-world accuracy, ONNX export or website deployment.


In [51]:
def smoke_tests():
    seed_everything(7)
    raw = np.zeros((CLIP_FRAMES,33,4),dtype=np.float32)
    raw[:,:,:3] = np.random.normal(0,0.3,(CLIP_FRAMES,33,3))
    x = torch.from_numpy(build_tensor(raw)).unsqueeze(0)
    assert x.shape == (1,7,CLIP_FRAMES,4,3) and torch.isfinite(x).all()
    for name in MODEL_NAMES:
        net=create_model(name)
        out=net(x)
        assert out.shape==(1,N_CLASSES) and torch.isfinite(out).all(),name
        out.square().mean().backward()
        assert any(p.grad is not None for p in net.parameters()),name
        print(name, 'OK | params=',count_parameters(net))
    # Synthetic dataset that mirrors your real 40 + 40 + 40 videos.
    synthetic=pd.DataFrame([{'video_id':f'{CLASS_NAMES[c]}/clip_{i:03d}',
                             'class_name':CLASS_NAMES[c], 'label':c}
                             for c in range(N_CLASSES) for i in range(40)])
    tests=[]
    folds=list(iter_stratified5_folds(synthetic, seed=13, max_folds=None))
    assert len(folds)==5
    for f in folds:
        train,val,test=set(f['train']),set(f['val']),set(f['test'])
        assert len(train)==81 and len(val)==15 and len(test)==24
        assert train.isdisjoint(val) and train.isdisjoint(test) and val.isdisjoint(test)
        assert set(synthetic.iloc[f['train']].label)==set(range(N_CLASSES))
        assert np.bincount(synthetic.iloc[f['test']].label.to_numpy(), minlength=N_CLASSES).tolist()==[8,8,8]
        assert np.bincount(synthetic.iloc[f['val']].label.to_numpy(), minlength=N_CLASSES).tolist()==[5,5,5]
        assert len(f['test_video_ids'])==len(f['test'])
        tests.extend(f['test'])
    assert sorted(tests)==list(range(len(synthetic)))
    # Segment-contract smoke test. Earliest qualifying 4-joint speed peak wins.
    fake = np.zeros((85, 33, 4), dtype=np.float32)
    # Dramatic jump at frame 60 (arms only), stable elsewhere.
    fake[60:, VELOCITY_LM, 0] = 3.0
    peak, _ = find_peak_velocity_frame(fake)
    assert abs(peak - 60) <= 2, peak  # moving-average plateau can shift ±2
    proxy = simulate_webcam_30_to_60(fake[:CLIP_FRAMES])
    assert proxy.shape == (CLIP_FRAMES,33,4) and np.isfinite(proxy).all()
    print('All smoke tests passed: peak selection, webcam proxy, 5 stratified folds, 120/120 test coverage, no overlap.')

# smoke_tests()

def audit_v25_data_files(manifest, n_per_class=2):
    print('v25 peak-window data compatibility audit (sampled files):')
    for label in CLASS_NAMES:
        sub=manifest.loc[manifest.class_name.eq(label)].head(n_per_class)
        if sub.empty:
            raise ValueError(f'Missing class directory/data: {label}')
        for row in sub.itertuples():
            raw=np.load(row.path)
            a=prepare_raw_clip(raw)
            f=build_tensor(a)
            assert a.shape==(CLIP_FRAMES,33,4), (row.path,a.shape)
            assert f.shape==(7,CLIP_FRAMES,4,3), (row.path,f.shape)
            assert np.isfinite(f).all(), f'Non-finite feature values: {row.path}'
            print(f'  {row.relative_path}: raw={tuple(raw.shape)} -> tensor={tuple(f.shape)}')
    print('All sampled clips follow the v25 data-shape contract.')

In [52]:
# No Drive videos required for this check; run after model/data setup.
smoke_tests()
# When segmentation has completed:
# manifest = build_manifest()
# check_folds(manifest)
# audit_v25_data_files(manifest)

dinat_large OK | params= 6623235
stgcn_large OK | params= 2744355
semgcn_large OK | params= 2796035
motionbert_large OK | params= 6270741
dastformer_large OK | params= 7612467
All smoke tests passed: peak selection, webcam proxy, 5 stratified folds, 120/120 test coverage, no overlap.


## 11. Run v25 Lite peak-based stratified five-fold CV — **five large models**

Set `RUN_EXPERIMENTS=True`, `N_SPLITS=5`, and `MAX_FOLDS=None` to train all five models across five folds (**25 training jobs**, up to `120` epochs each, early stopping enabled). The same held-out indices are shared across architectures. The output folder uses `stratified5_lite_peak_v25_large5` to prevent reusing stale checkpoints/results from previous model definitions. Keep `RUN_FINAL_REFIT=False` until all clean and webcam-proxy held-out results have been reviewed. This can require substantial GPU compute; no GPU is required just to run `smoke_tests()`.


In [53]:
if RUN_EXPERIMENTS:
    manifest = build_manifest()
    check_folds(manifest)
    audit_v25_data_files(manifest)
    predictions = evaluate_stratified5(manifest, MODELS_TO_RUN)
    summary = summarize_stratified5(predictions)
    proxy_predictions = pd.read_csv(os.path.join(RUN_DIR, 'stratified5_webcam_proxy_predictions.csv'))
    proxy_summary = summarize_stratified5(proxy_predictions, show_plots=False,
                 filename='stratified5_webcam_proxy_summary.csv')
    print('Standard held-out vs synthetic 30-FPS proxy:')
    print(summary[['model','pooled_macro_f1']].merge(
        proxy_summary[['model','pooled_macro_f1']], on='model',
        suffixes=('_clean','_webcam_proxy')).to_string(index=False))
    print('Results:', RUN_DIR)
else:
    print('Training disabled. Set RUN_EXPERIMENTS=True in config and rerun this cell to start.')

Training disabled. Set RUN_EXPERIMENTS=True in config and rerun this cell to start.


## 12. Final deployment model training (only after CV model selection)

Full five-fold evaluation of all **five large architectures** comes first. Review clean AND synthetic webcam-proxy macro-F1 plus per-class recall and confusion matrices before choosing `FINAL_MODEL_NAME`. Set `RUN_FINAL_REFIT=True` only after selecting your architecture. The refit uses all 120 labeled examples and the median best validation epoch from the 5 folds; its in-sample accuracy is *not* independent held-out accuracy. The JavaScript model needs a new ONNX export with matching input/output tensor names. For actual browser accuracy, run a separate webcam capture test.


In [54]:
def fit_final_model(manifest, model_name, epochs, filename=None):
    """Final refit on all available labeled videos after CV model selection."""
    if model_name not in MODEL_ZOO: raise ValueError(model_name)
    if int(epochs) < 1: raise ValueError('epochs must be positive')
    seed_everything(SEED + 50000)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = create_model(model_name).to(device)
    full_ds = BoxingDataset(manifest, augment=AUGMENT_TRAIN)
    labels = manifest.label.to_numpy(dtype=int)
    if SAMPLER_MODE == 'hook_x3':
        weights = np.where(labels==CLASS_NAMES.index('lead-hook'),3.,1.)
    elif SAMPLER_MODE == 'balanced':
        counts = np.bincount(labels,minlength=N_CLASSES)
        weights = 1. / np.maximum(counts[labels],1)
    elif SAMPLER_MODE == 'shuffle':
        weights = None
    else: raise ValueError(SAMPLER_MODE)
    sampler = WeightedRandomSampler(torch.as_tensor(weights,dtype=torch.double),
                                    num_samples=len(labels),replacement=True) if weights is not None else None
    loader = DataLoader(full_ds,batch_size=BATCH_SIZE,shuffle=sampler is None,
                        sampler=sampler,num_workers=NUM_WORKERS)
    optim = torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EPOCHS)
    for epoch in range(1,int(epochs)+1):
        metrics = run_one_epoch(model,loader,device,optim)
        scheduler.step()
        if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
            print(f'FINAL {model_name} ep={epoch}/{epochs}: acc={metrics["acc"]:.3f} loss={metrics["loss"]:.3f}')
    out = filename or os.path.join(RUN_DIR,f'FINAL_{model_name}.pt')
    torch.save({'state_dict':model.cpu().state_dict(),'model_name':model_name,
                'classes':CLASS_NAMES,'clip_frames':CLIP_FRAMES,'fps':TARGET_FPS,
                'n_channels':N_CHANNELS,'training_videos':len(manifest),'epochs':int(epochs),
                'augmentation_mode':AUGMENTATION_MODE, 'pose_variant': POSE_VARIANT,
                'preprocessing_contract':PREPROCESS_CONTRACT,
                'webcam_fps_aug_prob':WEBCAM_FPS_AUG_PROB, 'webcam_capture_fps':WEBCAM_CAPTURE_FPS,
                'role':'final_full_data_refit'},out)
    print('Final model saved to:',out)
    return out

# Example after FULL cross-validation:
# best_name = 'stgcn_large'   # use actual results
# cv = pd.read_csv(os.path.join(RUN_DIR,'stratified5_predictions.csv'))
# best_epochs = cv.loc[cv.model.eq(best_name),'best_epoch'].to_numpy()
# final_epochs = int(np.median(best_epochs))
# final_ckpt = fit_final_model(build_manifest(),best_name,final_epochs)


## 13. Offline real-video prediction and ONNX export

`predict_video()` uses the same MediaPipe Lite world-landmark normalization, 60-FPS resampling, **four-joint / five-frame smoothed peak**, 50-frame window, and seven-channel features as training (no augmentation at inference). `export_onnx()` exports input `skeleton_volume` `(B,7,50,4,3)` and output `class_logits` `(B,3)`. The team's current JS runtime instead expects `input`/`logits`; change JS inference keys or export names before integration. The ONNX architecture must be supported by the browser's selected ONNX Runtime Web provider, particularly for attention-based variants.


In [55]:
def load_checkpoint(checkpoint, device=None):
    device = device or torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    meta = torch.load(checkpoint, map_location="cpu", weights_only=True)

    assert meta["clip_frames"] == CLIP_FRAMES
    assert meta["classes"] == CLASS_NAMES
    assert meta.get("fps", TARGET_FPS) == TARGET_FPS
    assert meta.get("n_channels", 7) == N_CHANNELS

    contract = meta.get("preprocessing_contract")
    if contract != PREPROCESS_CONTRACT:
        raise ValueError(
            f"Wrong checkpoint preprocessing: expected {PREPROCESS_CONTRACT}, "
            f"got {contract}"
        )

    pose_variant = meta.get("pose_variant")
    if pose_variant is not None and pose_variant != POSE_VARIANT:
        raise ValueError(
            f"Checkpoint trained using {pose_variant} pose; "
            f"active preprocessing is {POSE_VARIANT}"
        )

    net = create_model(meta["model_name"]).to(device)
    net.load_state_dict(meta["state_dict"])
    return net.eval(), meta, device


@torch.no_grad()
def predict_video(video_path, checkpoint):
    net, meta, device = load_checkpoint(checkpoint)
    clip = extract_best_clip(video_path)
    if clip is None:
        return None

    x = torch.from_numpy(build_tensor(prepare_raw_clip(clip)))
    x = x.unsqueeze(0).to(device)
    probs = net(x).softmax(dim=1)[0].cpu().numpy()

    return {
        "predicted_label": CLASS_NAMES[int(probs.argmax())],
        "confidence": float(probs.max()),
        "probabilities": {
            name: float(prob) for name, prob in zip(CLASS_NAMES, probs)
        },
        "model": meta["model_name"],
    }


def export_onnx(checkpoint, filename=None, opset=17):
    net, meta, _ = load_checkpoint(checkpoint, device=torch.device("cpu"))
    out = filename or os.path.join(RUN_DIR, f"{meta['model_name']}_final.onnx")

    dummy = torch.zeros(
        1, N_CHANNELS, CLIP_FRAMES, N_ARM_JOINTS, 3,
        dtype=torch.float32,
    )

    torch.onnx.export(
        net,
        dummy,
        out,
        opset_version=opset,
        dynamo=False,
        input_names=["input"],
        output_names=["logits"],
    )

    print(f"ONNX saved: {out} ({os.path.getsize(out) / 1e6:.2f} MB)")
    return out

## 14. Important evaluation and deployment notes

- **Never reuse old processed `.npy` clips** from v21/v22/v23/v24. v25 has a new extraction folder and JSON preprocessing contract. Old reversal-based checkpoints are incompatible with the new peak-based distribution.
- Data augmentation (including the default random 30→60 FPS proxy) is **training-only**. Clean validation/test splits are untouched. The additional deterministic webcam-proxy test is held out by video but is still **synthetic**, not actual website accuracy.
- Whole-video extraction searches for one global speed peak after the 50-frame minimum. The website's `segmentClip(startT,endT)` first narrows the search window using a punch detector, and its `snapshotClip()` fallback may select different samples. Matching the core peak-velocity algorithm **does not guarantee identical boundaries**.
- `web/pipeline.js` needs the agreed **4-joint + 5-frame** settings; v25 feature channel 5 already uses the full 3D speed denominator. JavaScript currently uses `sqrt(vx²+vz²)` and must change to `sqrt(vx²+vy²+vz²)`. Verify all seven channels numerically using a saved skeleton clip.
- Keep 60 FPS as the resampling target, not a claim that the browser or webcam processes 60 real frames/second. The live JS code estimates average processing FPS, which can drift and obscure dropped/irregular frames.
- Your browser currently requires >=50 resampled frames and returns null otherwise; this offline notebook front-pads short videos. Avoid short clips in deployment or explicitly align the two policies.
- Stratified CV is **video-level, not boxer-disjoint**. It is not an estimate of performance on unseen people. Use a fresh person-disjoint webcam set to select the final model for real use.
- ONNX export default names are `skeleton_volume`/`class_logits`, whereas current website expects `input`/`logits`. Ensure names and provider support match the actual ONNX file loaded; do not just replace the 21 MB CNN weights with a different architecture.

- **New architectures:** The five model names and their weight shapes differ from previous versions. Do **not** reuse older model checkpoints. Compare parameters via `print_comparison_footprints()`; expect larger memory and longer fitting times. These are adapted classification models, not pretrained full-size DiNAT, MotionBERT or DASTFormer replications.
- **Opt-in execution:** Turn on `RUN_EXPERIMENTS` only when ready to run 25 expensive training jobs. `RUN_FINAL_REFIT` remains disabled until an architecture is selected from complete held-out validation.


## 15. Optional final full-data refit (no Gradio)

After completing all five video-level CV folds for each candidate, select `FINAL_MODEL_NAME` from `dinat_large`, `stgcn_large`, `semgcn_large`, `motionbert_large`, or `dastformer_large` and set `RUN_FINAL_REFIT=True`. Run this section, then export the selected **new** checkpoint with `export_onnx(final_ckpt)` in Section 13. No Gradio code or obsolete tiny/CNN model is included; the deployment target remains the existing JavaScript MediaPipe + ONNX Runtime Web stack. Browser runtime compatibility and model accuracy must be checked separately.


In [56]:
if RUN_FINAL_REFIT:
    # SECTION 15 — opt-in deployment refit AFTER checking complete v25 CV results
    import os
    import numpy as np
    import pandas as pd
    from torch.utils.data import DataLoader

    manifest = build_manifest()
    assert len(manifest) == 120, f"Expected the same 120-video dataset; got {len(manifest)}"
    cv_path = os.path.join(RUN_DIR, 'stratified5_predictions.csv')
    if os.path.isfile(cv_path):
        cv = pd.read_csv(cv_path)
        selected_folds = (cv.loc[cv['model'] == FINAL_MODEL_NAME, ['fold', 'best_epoch']]
                         .drop_duplicates(subset=['fold']).sort_values('fold'))
        assert selected_folds['fold'].nunique() == N_SPLITS, (
            f"Expected {N_SPLITS} completed v25 {FINAL_MODEL_NAME} folds, got {len(selected_folds)}. "
            f"Run all CV folds for this candidate, or remove stale {cv_path}."
        )
        FINAL_EPOCHS = max(1, int(round(selected_folds['best_epoch'].median())))
        print(f'Lite {FINAL_MODEL_NAME} CV best epochs:', selected_folds['best_epoch'].tolist())
        print(f'Lite {FINAL_MODEL_NAME} CV pooled held-out accuracy:',
              f"{(cv.loc[cv.model == FINAL_MODEL_NAME, 'true'] == cv.loc[cv.model == FINAL_MODEL_NAME, 'pred']).mean():.2%}")
    else:
        raise FileNotFoundError(f'No v25 CV predictions at {cv_path}; run all 5 folds first.')
    print('Selected final architecture:', FINAL_MODEL_NAME)
    print('Full-data training epochs:', FINAL_EPOCHS)
    print('Training config:', 'augmentation=', AUGMENTATION_MODE, 'sampler=', SAMPLER_MODE)

    final_ckpt = fit_final_model(
        manifest,
        model_name=FINAL_MODEL_NAME,
        epochs=FINAL_EPOCHS,
        filename=os.path.join(RUN_DIR, f'FINAL_{FINAL_MODEL_NAME}_full120.pt'),
    )

    # Only an IN-SAMPLE sanity check; this is NOT held-out generalization accuracy.
    net, meta, device = load_checkpoint(final_ckpt)
    training_loader = DataLoader(
        BoxingDataset(manifest, augment=False),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
    )
    training_metrics = run_one_epoch(net, training_loader, device)
    print(f"IN-SAMPLE final training accuracy: {training_metrics['acc']:.2%}")
    print(f"IN-SAMPLE final training macro-F1: {training_metrics['macro_f1']:.4f}")
    print('NOTE: In-sample final training scores are NOT generalization accuracy.')
    print('Final checkpoint:', final_ckpt)
else:
    print('CV-only run: inspect the v25 clean + webcam-proxy summaries; choose FINAL_MODEL_NAME, then enable RUN_FINAL_REFIT.')


Video counts: {'jab': 40, 'lead-hook': 40, 'rear-uppercut': 40} | Total: 120
Lite stgcn_large CV best epochs: [31, 46, 43, 51, 26]
Lite stgcn_large CV pooled held-out accuracy: 89.17%
Selected final architecture: stgcn_large
Full-data training epochs: 43
Training config: augmentation= v12 sampler= hook_x3
FINAL stgcn_large ep=1/43: acc=0.567 loss=0.970
FINAL stgcn_large ep=10/43: acc=0.767 loss=0.573
FINAL stgcn_large ep=20/43: acc=0.933 loss=0.331
FINAL stgcn_large ep=30/43: acc=0.958 loss=0.244
FINAL stgcn_large ep=40/43: acc=0.967 loss=0.235
FINAL stgcn_large ep=43/43: acc=0.975 loss=0.232
Final model saved to: /content/drive/MyDrive/newPerfectPunchData/stratified5_lite_peak_v25_large5/FINAL_stgcn_large_full120.pt
IN-SAMPLE final training accuracy: 99.17%
IN-SAMPLE final training macro-F1: 0.9917
NOTE: In-sample final training scores are NOT generalization accuracy.
Final checkpoint: /content/drive/MyDrive/newPerfectPunchData/stratified5_lite_peak_v25_large5/FINAL_stgcn_large_full12

In [57]:
onnx_path = export_onnx(
    final_ckpt,
    filename=os.path.join(RUN_DIR, "punch_model.onnx"),
)

/tmp/ipykernel_616/1630718871.py:61: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/tmp/ipykernel_616/3590141476.py:14: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if x.ndim != 5 or x.shape[1:] != (N_CHANNELS, CLIP_FRAMES, NODES, 3):


ONNX saved: /content/drive/MyDrive/newPerfectPunchData/stratified5_lite_peak_v25_large5/punch_model.onnx (11.02 MB)


## 16. JavaScript handoff checklist (website code **not modified**)

To align `perfect-punch-test-main/web/pipeline.js` with **this** notebook, the teammate should update:

```javascript
export const CLIP_FRAMES = 50;
export const TARGET_FPS = 60;
export const VEL_WINDOW = 5;                 // formerly 7
const VELOCITY_LM = [13, 14, 15, 16];       // formerly included shoulders
// ch5: vx / (Math.hypot(vx, vy, vz) + 1e-8)
```

Keep `resampleFrames()` and the 50-frame window *ending at and including* the smoothed peak. Verify the website's punch detector window selection does not truncate the true peak or omit the required lead-in. JS currently uses `segmentClip()` and a different fallback; do not assume it matches an offline whole-video peak search.

To integrate the exported ONNX model, update `game.js`'s ONNX input/output names to match `skeleton_volume` / `class_logits`, or explicitly export with `input` / `logits`. Preserve class ordering. Test the output on exactly the same saved 50-frame skeleton tensor in Python and JS before evaluating webcam performance. The JS website ZIP is **not edited** by this notebook.
